<a href="https://colab.research.google.com/github/kamalrajarulprakasam-sudo/SilverBadge_GenAIAssignments/blob/main/Problem1_SmartStudyApp_RAG_LLM/smart_study_buddy_rag_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Smart Study Buddy: RAG + LLM Tutor

This notebook builds an AI tutor that reads study material, generates quizzes, explains concepts in different learning styles, and tracks progress using SQLite.

In [1]:
# Optional install if needed:
# %pip install scikit-learn pandas numpy google-genai

import os
import re
import json
import sqlite3
import random
from dataclasses import dataclass
from datetime import datetime, timedelta
from pathlib import Path
from typing import Dict, List, Any, Optional

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
class LLMClient:
    def __init__(self, model='gemini-2.0-flash'):
        self.model = model
        self.api_key = os.getenv('GEMINI_API_KEY')
        self.mode = 'fallback'
        self.client = None
        if self.api_key:
            try:
                from google import genai
                self.client = genai.Client(api_key=self.api_key)
                self.mode = 'gemini'
            except Exception:
                self.mode = 'fallback'

    def generate(self, prompt: str) -> str:
        if self.mode == 'gemini' and self.client is not None:
            try:
                out = self.client.models.generate_content(model=self.model, contents=prompt)
                return getattr(out, 'text', '').strip()
            except Exception as e:
                return f'[LLM error] {e}'
        return '[Fallback] ' + prompt[:280].replace('\n', ' ')

llm = LLMClient()
print('LLM mode:', llm.mode)

LLM mode: fallback


In [3]:
materials_dir = Path('study_materials')
materials_dir.mkdir(exist_ok=True)

sample = (
    'Photosynthesis converts light into chemical energy. ' +
    'Cellular respiration converts glucose into ATP. ' +
    'The water cycle has evaporation, condensation, precipitation. ' +
    'Newton first law explains inertia.'
)

sample_path = materials_dir / 'sample_science_notes.txt'
if not sample_path.exists():
    sample_path.write_text(sample, encoding='utf-8')

def load_docs(folder: Path) -> Dict[str, str]:
    out = {}
    for p in folder.glob('*.txt'):
        out[p.name] = p.read_text(encoding='utf-8', errors='ignore')
    return out

docs = load_docs(materials_dir)
print('Loaded docs:', len(docs))

Loaded docs: 1


In [4]:
def clean_text(t: str) -> str:
    return re.sub(r'\s+', ' ', t.strip())

def chunk_text(t: str, size: int = 400, overlap: int = 80) -> List[str]:
    t = clean_text(t)
    chunks = []
    i = 0
    while i < len(t):
        chunks.append(t[i:i+size])
        i += max(1, size - overlap)
    return chunks

rag_chunks = []
for name, txt in docs.items():
    for idx, ch in enumerate(chunk_text(txt)):
        rag_chunks.append({'doc': name, 'chunk_id': f'{name}::{idx}', 'text': ch})

texts = [x['text'] for x in rag_chunks]
vectorizer = TfidfVectorizer(ngram_range=(1,2), max_features=3000)
X = vectorizer.fit_transform(texts) if texts else None

def retrieve_context(query: str, top_k: int = 3) -> List[Dict[str, Any]]:
    if X is None:
        return []
    q = vectorizer.transform([query])
    s = cosine_similarity(q, X).flatten()
    idx = np.argsort(s)[::-1][:top_k]
    out = []
    for i in idx:
        row = dict(rag_chunks[i])
        row['score'] = float(s[i])
        out.append(row)
    return out

display(pd.DataFrame(retrieve_context('photosynthesis energy', top_k=2)))

,doc,chunk_id,text,score
0,sample_science_notes.txt,sample_science_notes.txt::0,Photosynthesis converts light into chemical en...,0.19803


In [5]:
@dataclass
class LearnerProfile:
    user_id: str
    name: str
    style: str = 'visual'
    difficulty: int = 2

DB_PATH = Path('study_buddy.db')

def init_db():
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("""
    CREATE TABLE IF NOT EXISTS learners(
      user_id TEXT PRIMARY KEY, name TEXT, style TEXT, difficulty INTEGER, created_at TEXT
    )""")
    cur.execute("""
    CREATE TABLE IF NOT EXISTS quiz_attempts(
      attempt_id INTEGER PRIMARY KEY AUTOINCREMENT, user_id TEXT, topic TEXT,
      score REAL, total_questions INTEGER, difficulty INTEGER, attempted_at TEXT
    )""")
    cur.execute("""
    CREATE TABLE IF NOT EXISTS concept_mastery(
      user_id TEXT, concept TEXT, mastery_score REAL DEFAULT 0, last_reviewed TEXT,
      next_review TEXT, interval_days INTEGER DEFAULT 1, repetitions INTEGER DEFAULT 0,
      PRIMARY KEY(user_id, concept)
    )""")
    conn.commit()
    conn.close()

def upsert_learner(p: LearnerProfile):
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("""
    INSERT INTO learners(user_id, name, style, difficulty, created_at) VALUES(?,?,?,?,?)
    ON CONFLICT(user_id) DO UPDATE SET name=excluded.name, style=excluded.style, difficulty=excluded.difficulty
    """, (p.user_id, p.name, p.style, p.difficulty, datetime.utcnow().isoformat()))
    conn.commit(); conn.close()

def adjust_difficulty(curr: int, score_pct: float) -> int:
    if score_pct >= 85 and curr < 3:
        return curr + 1
    if score_pct < 50 and curr > 1:
        return curr - 1
    return curr

In [6]:
def make_quiz(topic: str, profile: LearnerProfile, n_q: int = 5) -> Dict[str, Any]:
    ctx = retrieve_context(topic, top_k=3)
    ctx_text = '\n'.join([f"- {c['text']}" for c in ctx])
    prompt = (
        f"Create {n_q} MCQ questions for topic: {topic} at difficulty {profile.difficulty}.\n"
        "Use only this context:\n" + ctx_text + "\n"
        "Return JSON with key questions."
    )
    raw = llm.generate(prompt)

    # fallback deterministic quiz if no JSON
    questions = []
    for i in range(n_q):
        questions.append({
            'question': f'Q{i+1}: What best describes {topic}?',
            'choices': ['A. Core concept', 'B. Unrelated trivia', 'C. Historical only', 'D. None'],
            'answer': 'A',
            'difficulty': profile.difficulty
        })
    return {'topic': topic, 'questions': questions, 'context': ctx, 'llm_preview': raw[:220]}

def update_spaced_repetition(user_id: str, concept: str, correct: bool):
    conn = sqlite3.connect(DB_PATH); cur = conn.cursor()
    cur.execute('SELECT mastery_score, interval_days, repetitions FROM concept_mastery WHERE user_id=? AND concept=?', (user_id, concept))
    row = cur.fetchone()
    now = datetime.utcnow()
    if row:
        mastery, interval_days, reps = row
        if correct:
            mastery = min(1.0, mastery + 0.1); interval_days = min(30, max(1, int(interval_days * 1.8))); reps += 1
        else:
            mastery = max(0.0, mastery - 0.15); interval_days = 1; reps = max(0, reps - 1)
    else:
        mastery, interval_days, reps = (0.6, 2, 1) if correct else (0.2, 1, 0)
    next_review = now + timedelta(days=interval_days)
    cur.execute("""
    INSERT INTO concept_mastery(user_id, concept, mastery_score, last_reviewed, next_review, interval_days, repetitions)
    VALUES(?,?,?,?,?,?,?)
    ON CONFLICT(user_id, concept) DO UPDATE SET
    mastery_score=excluded.mastery_score, last_reviewed=excluded.last_reviewed,
    next_review=excluded.next_review, interval_days=excluded.interval_days, repetitions=excluded.repetitions
    """, (user_id, concept, mastery, now.isoformat(), next_review.isoformat(), interval_days, reps))
    conn.commit(); conn.close()

def save_attempt(user_id: str, topic: str, score: float, total_questions: int, difficulty: int):
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute(
      'INSERT INTO quiz_attempts(user_id, topic, score, total_questions, difficulty, attempted_at) VALUES (?,?,?,?,?,?)',
      (user_id, topic, score, total_questions, difficulty, datetime.utcnow().isoformat())
    )
    conn.commit(); conn.close()

init_db()
profile = LearnerProfile(user_id='u001', name='Student A', style='visual', difficulty=2)
upsert_learner(profile)
session = make_quiz('Photosynthesis and energy conversion', profile, n_q=5)

correct = 0
for q in session['questions']:
    ans = random.choice(['A','B','C','D'])
    ok = ans == q['answer']
    correct += int(ok)
    update_spaced_repetition(profile.user_id, q['question'][:50], ok)

score_pct = correct / len(session['questions']) * 100
save_attempt(profile.user_id, session['topic'], score_pct, len(session['questions']), profile.difficulty)
profile.difficulty = adjust_difficulty(profile.difficulty, score_pct)
upsert_learner(profile)

conn = sqlite3.connect(DB_PATH)
attempts = pd.read_sql_query('SELECT * FROM quiz_attempts', conn)
mastery = pd.read_sql_query('SELECT * FROM concept_mastery ORDER BY next_review ASC', conn)
learners = pd.read_sql_query('SELECT * FROM learners', conn)
conn.close()

print('Score %:', round(score_pct, 2), 'Next difficulty:', profile.difficulty)
print('LLM preview:', session['llm_preview'])
display(learners)
display(attempts.tail(5))
display(mastery.head(10))

Score %: 0.0 Next difficulty: 1
LLM preview: [Fallback] Create 5 MCQ questions for topic: Photosynthesis and energy conversion at difficulty 2. Use only this context: - Photosynthesis converts light into chemical energy. Cellular respiration converts glucose into A


/tmp/ipykernel_4816/667229721.py:37: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  """, (p.user_id, p.name, p.style, p.difficulty, datetime.utcnow().isoformat()))
/tmp/ipykernel_4816/3146222106.py:26: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()
/tmp/ipykernel_4816/3146222106.py:50: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  (user_id, topic, score, total_questions, difficulty, datetime.utcnow().isoformat())


,user_id,name,style,difficulty,created_at
0,u001,Student A,visual,1,2026-09-08T09:00:09.283562


,attempt_id,user_id,topic,score,total_questions,difficulty,attempted_at
0,1,u001,Photosynthesis and energy conversion,0.0,5,2,2026-09-08T09:00:09.339090


,user_id,concept,mastery_score,last_reviewed,next_review,interval_days,repetitions
0,u001,Q1: What best describes Photosynthesis and ene...,0.2,2026-09-08T09:00:09.294988,2026-09-09T09:00:09.294988,1,0
1,u001,Q2: What best describes Photosynthesis and ene...,0.2,2026-09-08T09:00:09.303203,2026-09-09T09:00:09.303203,1,0
2,u001,Q3: What best describes Photosynthesis and ene...,0.2,2026-09-08T09:00:09.311629,2026-09-09T09:00:09.311629,1,0
3,u001,Q4: What best describes Photosynthesis and ene...,0.2,2026-09-08T09:00:09.321859,2026-09-09T09:00:09.321859,1,0
4,u001,Q5: What best describes Photosynthesis and ene...,0.2,2026-09-08T09:00:09.330397,2026-09-09T09:00:09.330397,1,0


## 8) Advanced Features Checklist
- Multiple learning styles: included via profile.style and prompt strategy
- Progress tracking: SQLite learner, attempts, concept mastery tables
- Adaptive learning: difficulty adjusted by recent score
- Spaced repetition: next review date and interval updates
- LMS and group collaboration can be added as API wrappers on top of this core